# TAG Construction & Training — Main Interface

In [1]:
import sys, importlib
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import networkx as nx
import torch

import generic_data_manager, variant_registry, edge_factory
import label_factory, tag_constructor, models, trainer
for mod in [generic_data_manager, variant_registry, edge_factory,
            label_factory, tag_constructor, models, trainer]:
    importlib.reload(mod)

from generic_data_manager import GenericDataManager
from variant_registry import VariantRegistry
from tag_constructor import TAGConstructor
from models import ModelFactory
from trainer import GNNTrainer, build_result_row, RESULT_COLUMNS

## Config

In [2]:
DATASET      = 'arxiv'   # 'arxiv' | 'amazon' | 'history'
SPLIT        = 'train'
DATA_ROOT    = '../data'
HIDDEN_DIM   = 256
DROPOUT      = 0.5
LR           = 0.001
EPOCHS       = 200
PATIENCE     = 50
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cpu


## 1. Initialise managers

In [3]:
dm       = GenericDataManager(DATASET, base_path=DATA_ROOT)
registry = VariantRegistry(DATASET, config_path=f'{DATA_ROOT}/configs')
builder  = TAGConstructor(dm)
print('Dataset config:', dm.get_config())

Dataset config: {'dataset': 'arxiv', 'has_secondary_id': True, 'has_structural_edges': False, 'embedding_prefix': 'article', 'contextual_embedding_available': True, 'secondary_embedding_prefix': 'author', 'aggregate_embedding_prefix': 'category'}


## 2. Variant space

In [4]:
all_variants = list(registry.enumerate_variants())
# M1-M4 only (M5/M6 require multi-graph batching — see PENDING_M5_M6.md)
train_variants = [v for v in all_variants if v['M'] in ('M1','M2','M3','M4')]
print(f'Total variants: {len(all_variants)}  |  M1-M4 trainable: {len(train_variants)}')
pd.DataFrame(train_variants).value_counts('M').sort_index()

Total variants: 180  |  M1-M4 trainable: 84


M
M1    24
M2    48
M3     6
M4     6
Name: count, dtype: int64

## 3. Load a sample

In [5]:
SAMPLE_IDX = 0
df_sample  = dm.load_data(SPLIT, sample_idx=SAMPLE_IDX)
print(f'Loaded {len(df_sample)} rows')
df_sample.head(3)

Loaded 1000 rows


,primary_id,secondary_id,aggregate_id,text_fidelity_a,text_fidelity_b,categorical_label,scalar_label,structural_edges
0,711.0666,"[Ghazi|Bouselmi, Dominique|Fohr, Irina|Illina,...",cs,Discriminative Phoneme Sequences Extraction fo...,Discriminative Phoneme Sequences Extraction fo...,cs,NaN,NaN
1,804.4052,[Johannes|Sjoestrand],math,Eigenvalue distributions and Weyl laws for sem...,Eigenvalue distributions and Weyl laws for sem...,math,NaN,NaN
2,805.2670,[Syksy|Rasanen],astro-ph,The effect of structure formation on the expan...,The effect of structure formation on the expan...,astro-ph,NaN,NaN


## 4. Construct & inspect a single TAG

In [6]:
variant = train_variants[0]
print('Variant:', variant)
data = builder.construct(variant, df_sample, SPLIT)
print(data)
print(f'  nodes  : {data.num_nodes}')
print(f'  edges  : {data.edge_index.shape[1]//2} undirected')
print(f'  x      : {tuple(data.x.shape)}')
print(f'  y      : {tuple(data.y.shape)}')
for attr in ('train_mask','val_mask','test_mask','target_edge_index'):
    if hasattr(data, attr):
        v = getattr(data, attr)
        print(f'  {attr}: {tuple(v.shape)}')

Variant: {'M': 'M1', 'N': 'N7', 'E': 'E10b', 'T': 'T12a'}
Data(num_nodes=1000, x=[1000, 384], y=[1000, 18], edge_index=[2, 98], train_mask=[1000], val_mask=[1000], test_mask=[1000])
  nodes  : 1000
  edges  : 49 undirected
  x      : (1000, 384)
  y      : (1000, 18)
  train_mask: (1000,)
  val_mask: (1000,)
  test_mask: (1000,)


## 5. Train a single variant (quick sanity check)

In [7]:
M_TO_TASK = {'M1':'categorical','M2':'scalar','M3':'edge_categorical','M4':'edge_scalar'}

variant    = train_variants[0]
task_type  = M_TO_TASK[variant['M']]
data       = builder.construct(variant, df_sample, SPLIT)

out_dim    = data.y.shape[-1] if variant['M'] == 'M1' else (2 if variant['M']=='M3' else 1)
in_dim     = data.x.shape[1]

model   = ModelFactory.create(task_type, in_dim, HIDDEN_DIM, out_dim, DROPOUT)
t       = GNNTrainer(model, device=DEVICE, lr=LR, epochs=EPOCHS,
                     patience=PATIENCE, task_type=task_type)

print(f'Training {variant}  |  task={task_type}  |  out_dim={out_dim}')
metrics = t.train(data, num_classes=out_dim, verbose=True, early_stopping=True)
print('\nTest metrics:', metrics)

row = build_result_row(variant, metrics, data, df_sample, t,
                       run_split=SPLIT, sample_idx=SAMPLE_IDX,
                       output_dimension=out_dim)
pd.Series(row)

Training {'M': 'M1', 'N': 'N7', 'E': 'E10b', 'T': 'T12a'}  |  task=categorical  |  out_dim=18
Epoch 001 | Loss 3.0184 | Train KL 2.9034 | Val KL 2.8997 | Val top1 0.073 | Val cos 0.233
Epoch 010 | Loss 2.4383 | Train KL 2.7812 | Val KL 2.7817 | Val top1 0.213 | Val cos 0.263
Epoch 020 | Loss 2.2363 | Train KL 2.5744 | Val KL 2.6027 | Val top1 0.213 | Val cos 0.309
Epoch 030 | Loss 1.9084 | Train KL 2.5047 | Val KL 2.5971 | Val top1 0.213 | Val cos 0.308
Epoch 040 | Loss 1.6160 | Train KL 2.3317 | Val KL 2.6138 | Val top1 0.253 | Val cos 0.306
Epoch 050 | Loss 1.3507 | Train KL 1.9300 | Val KL 2.6288 | Val top1 0.233 | Val cos 0.303
Epoch 060 | Loss 1.0683 | Train KL 1.2776 | Val KL 2.6383 | Val top1 0.193 | Val cos 0.306
Epoch 070 | Loss 0.8684 | Train KL 0.7238 | Val KL 2.8281 | Val top1 0.207 | Val cos 0.294
Early stopping at epoch 74

Test results: {'kl': 2.5694987773895264, 'top1': 0.22666666666666666, 'top3': 0.5866666666666667, 'cosine': 0.31625115871429443}


Test metrics: {'kl'

Task_Idx                          M1
Node_Idx                          N7
Edge_Idx                        E10b
Text_Idx                        T12a
task_type                categorical
KL                          2.569499
Top1                        0.226667
Top3                        0.586667
Cosine                      0.316251
Accuracy                    0.226667
F1                               NaN
MAE                              NaN
MSE                              NaN
R2                               NaN
Primary_Metric              accuracy
Primary_Value               0.226667
Performance_Band                None
normalized_score           22.666667
number_of_nodes                 1000
avg_text_length              130.415
text_vocab_entropy          7.250427
label_balance_entropy       2.365527
output_dimension                  18
run_split                      train
degenerate                     False
dtype: object

## 6. Batch training — M1 tasks only
Trains all M1 (node categorical classification) variants on one sample. Results are saved to `../output/quick_run_{DATASET}.csv`.

In [ ]:
import os, math
from pathlib import Path

# ── knobs ─────────────────────────────────────────────────────────────────
N_VARIANTS_CAP = None   # set to e.g. 10 to limit; None = all M1-M4 variants
QUICK_EPOCHS   = 50     # lower for speed; bump to 200 for full runs
QUICK_SAMPLE   = 0      # which sample_idx to use for this sweep
OUT_CSV        = f'../output/quick_run_{DATASET}.csv'
# ──────────────────────────────────────────────────────────────────────────

Path('../output').mkdir(exist_ok=True)

# Filter for M1 tasks only
m1_variants = [v for v in train_variants if v['M'] == 'M1']
variants_to_run = m1_variants if N_VARIANTS_CAP is None else m1_variants[:N_VARIANTS_CAP]

print(f'M1 variants to run: {len(variants_to_run)} (out of {len(m1_variants)} total M1 variants)')

df_s = dm.load_data(SPLIT, sample_idx=QUICK_SAMPLE)

rows = []
for i, v in enumerate(variants_to_run):
    M, N, E, T = v['M'], v['N'], v['E'], v['T']
    task_type  = M_TO_TASK[M]
    try:
        d       = builder.construct(v, df_s, SPLIT)
        out_dim = d.y.shape[-1] if M == 'M1' else (2 if M == 'M3' else 1)
        in_dim  = d.x.shape[1]

        mdl = ModelFactory.create(task_type, in_dim, HIDDEN_DIM, out_dim, DROPOUT)
        tr  = GNNTrainer(mdl, device=DEVICE, lr=LR, epochs=QUICK_EPOCHS,
                         patience=PATIENCE, task_type=task_type)
        met = tr.train(d, num_classes=out_dim, verbose=False, early_stopping=True)

        row = build_result_row(v, met, d, df_s, tr,
                               run_split=SPLIT, sample_idx=QUICK_SAMPLE,
                               output_dimension=out_dim)
        rows.append(row)
        pm, pv = row['Primary_Metric'], row['Primary_Value']
        print(f"[{i+1}/{len(variants_to_run)}] {M}/{N}/{E}/{T} | {pm}={pv:.4f} | degen={row['degenerate']}")
    except Exception as e:
        print(f"[{i+1}] ERROR {M}/{N}/{E}/{T}: {e}")
        import traceback; traceback.print_exc()

df_results = pd.DataFrame(rows, columns=RESULT_COLUMNS + ['sample_idx']) if rows else pd.DataFrame()
if not df_results.empty:
    df_results.to_csv(OUT_CSV, index=False)
    print(f'\nSaved {len(df_results)} rows → {OUT_CSV}')
df_results

## Smoke Test — M1-M6 pipeline validation
<!-- Run first 2 variants of each task type for history dataset, 2 samples each -->

In [12]:
import importlib
import sys
sys.path.insert(0, '.')

# reload modules
for mod_name in ['generic_data_manager', 'variant_registry', 'edge_factory',
                  'label_factory', 'tag_constructor', 'models', 'trainer', 'global_trainer']:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

from generic_data_manager import GenericDataManager
from variant_registry import VariantRegistry
from tag_constructor import TAGConstructor
from models import ModelFactory
from trainer import GNNTrainer, build_result_row
from global_trainer import GlobalTrainer, build_result_row_global

SMOKE_DATASET = 'history'
SMOKE_EPOCHS  = 5
SMOKE_SAMPLES = 2

dm_s       = GenericDataManager(SMOKE_DATASET, base_path='../data')
registry_s = VariantRegistry(SMOKE_DATASET, config_path='../data/configs')
builder_s  = TAGConstructor(dm_s)

all_variants_s = list(registry_s.enumerate_variants())

# Get first 2 variants per task type
from collections import defaultdict
by_M = defaultdict(list)
for v in all_variants_s:
    by_M[v['M']].append(v)

smoke_variants = []
for M in ['M1', 'M2', 'M3', 'M4', 'M5', 'M6']:
    smoke_variants.extend(by_M[M][:2])

print(f"Smoke testing {len(smoke_variants)} variants x {SMOKE_SAMPLES} train + {SMOKE_SAMPLES} test samples")
errors = []

for v in smoke_variants:
    M = v['M']
    task_type = {'M1': 'categorical', 'M2': 'scalar', 'M3': 'edge_categorical',
                 'M4': 'edge_scalar', 'M5': 'global_categorical', 'M6': 'global_scalar'}[M]
    try:
        if M in ('M5', 'M6'):
            # Global path
            train_graphs = [builder_s.construct(v, dm_s.load_data('train', i), 'train')
                            for i in range(SMOKE_SAMPLES)]
            test_graphs  = [builder_s.construct(v, dm_s.load_data('test',  i), 'test')
                            for i in range(SMOKE_SAMPLES)]
            out_dim = 2 if M == 'M5' else 1
            in_dim  = train_graphs[0].x.shape[1]
            model   = ModelFactory.create(task_type, in_dim, 64, out_dim, 0.5)
            gt      = GlobalTrainer(model, task_type, 'cpu', epochs=SMOKE_EPOCHS)
            gt.train(train_graphs)
            for g in train_graphs + test_graphs:
                r = gt.evaluate_single(g)
            print(f"  PASS {v}")
        else:
            # Node/edge path
            out_dim_map = {'M1': None, 'M2': 1, 'M3': 2, 'M4': 1}
            for split in ['train', 'test']:
                for si in range(SMOKE_SAMPLES):
                    df_s   = dm_s.load_data(split, si)
                    data_s = builder_s.construct(v, df_s, split)
                    odim   = data_s.y.shape[-1] if M == 'M1' else out_dim_map[M]
                    in_dim = data_s.x.shape[1]
                    model  = ModelFactory.create(task_type, in_dim, 64, odim, 0.5)
                    tr_s   = GNNTrainer(model, 'cpu', epochs=SMOKE_EPOCHS, task_type=task_type)
                    tr_s.train(data_s, num_classes=odim, verbose=False)
            print(f"  PASS {v}")
    except Exception as exc:
        import traceback
        errors.append((v, str(exc)))
        print(f"  FAIL {v}: {exc}")
        traceback.print_exc()

if errors:
    print(f"SMOKE TEST FAILED: {len(errors)} errors")
    for v, e in errors:
        print(f"  {v}: {e}")
else:
    print("SMOKE TEST PASSED — M1-M6 pipeline validated")


Smoke testing 12 variants x 2 train + 2 test samples
  PASS {'M': 'M1', 'N': 'N7', 'E': 'E10b', 'T': 'T12a'}
  PASS {'M': 'M1', 'N': 'N7', 'E': 'E10b', 'T': 'T12b'}
  PASS {'M': 'M2', 'N': 'N7', 'E': 'E10a', 'T': 'T12a'}
  PASS {'M': 'M2', 'N': 'N7', 'E': 'E10a', 'T': 'T12b'}
  PASS {'M': 'M3', 'N': 'N7', 'E': 'E10b', 'T': 'T12a'}
  PASS {'M': 'M3', 'N': 'N7', 'E': 'E10b', 'T': 'T12b'}
  PASS {'M': 'M4', 'N': 'N7', 'E': 'E10b', 'T': 'T12a'}
  PASS {'M': 'M4', 'N': 'N7', 'E': 'E10b', 'T': 'T12b'}
  PASS {'M': 'M5', 'N': 'N7', 'E': 'E10a', 'T': 'T12a'}
  PASS {'M': 'M5', 'N': 'N7', 'E': 'E10a', 'T': 'T12b'}
  PASS {'M': 'M6', 'N': 'N7', 'E': 'E10a', 'T': 'T12a'}
  PASS {'M': 'M6', 'N': 'N7', 'E': 'E10a', 'T': 'T12b'}
SMOKE TEST PASSED — M1-M6 pipeline validated


## Full Training Run — All Datasets, All Variants

In [14]:
from datetime import datetime
import sys
sys.path.insert(0, '.')

from experiment_runner import ExperimentRunner

RUN_TS  = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = f'../logs/run_{RUN_TS}/analysis'

print(f"Run directory: {RUN_DIR}")
print("Starting full run — this will take several hours.")
print("Progress streams to output CSV in real time; safe to interrupt and resume.")

runner = ExperimentRunner(
    base_path='../data',
    output_path=RUN_DIR,
    hidden_dim=256,
    epochs=100,
    lr=0.001,
    early_stopping=True,
    verbose=False,
)
runner.run()
print(f"Full run complete. Results in {RUN_DIR}/")


Run directory: ../logs/run_20260616_005642/analysis
Starting full run — this will take several hours.
Progress streams to output CSV in real time; safe to interrupt and resume.
2026-06-16 00:56:42,149 INFO ============================================================
2026-06-16 00:56:42,150 INFO Dataset: arxiv
2026-06-16 00:56:42,150 INFO ============================================================
2026-06-16 00:56:42,167 INFO Resuming from 0 existing rows in ../logs/run_20260616_005642/analysis/construction_performance_table_arxiv.csv
2026-06-16 00:56:42,169 INFO [1/180] arxiv | M1/N7/E10b/T12a | task=categorical
2026-06-16 00:56:43,744 INFO   train sample_00 | primary=accuracy=0.2267 | norm=22.7 | degen=False
2026-06-16 00:56:45,191 INFO   train sample_01 | primary=accuracy=0.1967 | norm=19.7 | degen=False
2026-06-16 00:56:46,577 INFO   train sample_02 | primary=accuracy=0.1600 | norm=16.0 | degen=False
2026-06-16 00:56:47,986 INFO   train sample_03 | primary=accuracy=0.1833 | norm=18

/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


2026-06-16 01:02:30,870 INFO   train sample_00 | primary=accuracy=0.2367 | norm=23.7 | degen=False


/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


2026-06-16 01:02:32,216 INFO   train sample_01 | primary=accuracy=0.1867 | norm=18.7 | degen=False


...

/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


2026-06-16 05:56:01,359 INFO   test sample_18 | primary=accuracy=0.8848 | norm=88.5 | degen=False


/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/tag_graphs/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


2026-06-16 05:56:13,624 INFO   test sample_19 | primary=accuracy=0.8682 | norm=86.8 | degen=False
2026-06-16 05:56:13,626 INFO [22/180] arxiv | M1/N8/E11b/T12a | task=categorical
2026-06-16 06:13:34,234 INFO   train sample_00 | primary=accuracy=0.8630 | norm=86.3 | degen=False
2026-06-16 06:33:24,965 INFO   train sample_01 | primary=accuracy=0.8369 | norm=83.7 | degen=False
2026-06-16 06:51:35,710 INFO   train sample_02 | primary=accuracy=0.8537 | norm=85.4 | degen=False
2026-06-16 07:14:46,189 INFO   train sample_03 | primary=accuracy=0.8567 | norm=85.7 | degen=False
2026-06-16 07:34:30,341 INFO   train sample_04 | primary=accuracy=0.8495 | norm=84.9 | degen=False
2026-06-16 07:49:41,838 INFO   train sample_05 | primary=accuracy=0.8057 | norm=80.6 | degen=False
2026-06-16 08:10:47,261 INFO   train sample_06 | primary=accuracy=0.8797 | norm=88.0 | degen=False
2026-06-16 08:30:48,680 INFO   train sample_07 | primary=accuracy=0.8520 | norm=85.2 | degen=False
2026-06-16 09:22:36,769 INFO 

KeyboardInterrupt: 

## 7. Results summary

In [ ]:
if not df_results.empty:
    summary = df_results.groupby('Task_Idx').agg(
        n=('Primary_Value','count'),
        primary_metric=('Primary_Metric','first'),
        mean_primary=('Primary_Value', lambda x: x.dropna().mean()),
        mean_norm_score=('normalized_score', lambda x: x.dropna().mean()),
        pct_degenerate=('degenerate', lambda x: 100*x.mean()),
    ).round(4)
    display(summary)

    # Best and worst per task
    for M in df_results['Task_Idx'].unique():
        sub = df_results[df_results['Task_Idx']==M].copy()
        sub = sub.dropna(subset=['Primary_Value'])
        if sub.empty: continue
        best = sub.loc[sub['Primary_Value'].idxmax()]
        print(f'{M} best  : {best["Node_Idx"]}/{best["Edge_Idx"]}/{best["Text_Idx"]}  {best["Primary_Metric"]}={best["Primary_Value"]:.4f}')